In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib

In [ ]:
train_df=pd.read_csv("./data/bank_app_reviews_train.csv")
train_df.head()

In [ ]:
test_df=pd.read_csv("./data/bank_app_reviews_test.csv")
test_df.head()

In [ ]:
train_df.info()

In [ ]:
test_df.info()

In [ ]:
train_df.columns

In [ ]:
train_df['은행명'].value_counts()

In [ ]:
train_df['평점'].value_counts()

In [ ]:
train_df[train_df['평점']==4]['사용자리뷰'].head(60)

In [ ]:
train_df[train_df['평점']==3]['사용자리뷰'].head(60)

In [ ]:
train_df['긍정']=train_df['평점'].apply(lambda x: 1 if x in [5, 4] else 0)
train_df

In [ ]:
test_df['긍정']=test_df['평점'].apply(lambda x: 1 if x in [5, 4] else 0)
test_df

# 변수를 자동으로 생성하면서 긍정부정 데이터프레임 만들기
* globals()[변수명]

In [ ]:
bank_en_list=['hana','shinhan', 'kb','toss','woori','b_salad']

In [ ]:
for bank, bank_eng in zip(train_df['은행명'].unique(), bank_en_list):
    print(bank, bank_eng)

In [ ]:
toss_pos=train_df[(train_df['은행명']=='토스')&(train_df['긍정']==1)]

In [ ]:
toss_neg=train_df[(train_df['은행명']=='토스')&(train_df['긍정']==0)]

In [ ]:
pos_neg_list=[]
for bank, bank_eng in zip(train_df['은행명'].unique(), bank_en_list):
    pos=train_df[(train_df['은행명']==bank)&(train_df['긍정']==1)]
    neg=train_df[(train_df['은행명']==bank)&(train_df['긍정']==0)]
    # 전역변수를 자동생성해서 저장
    globals()[f"{bank_eng}_pos"]=pos
    globals()[f"{bank_eng}_neg"]=neg
    pos_neg_list.append(f"{bank_eng}_pos")
    pos_neg_list.append(f"{bank_eng}_neg")
    #시각화
    plt.bar(['긍정','부정'],[pos.shape[0], neg.shape[0]], color=['skyblue','salmon'])
    plt.title(f'{bank} 긍부정 분포')
    plt.ylabel('리뷰수')
    plt.ylim(0, max(pos.shape[0], neg.shape[0] + 100))
    plt.show()
    print()
    print(pos.shape[0], neg.shape[0])

In [ ]:
pos_neg_list

In [ ]:
hana_pos

In [ ]:
for bank in pos_neg_list:
    display(globals()[bank])

모든 데이터 프레임에서 특수문자 제거하기

In [ ]:
import re
def text_clean(x):
    pattern=r'[가-힣0-9a-zA-Z]+'
    matches=re.findall(pattern, x)
    matches=" ".join(matches)
    return matches

In [ ]:
for bank in pos_neg_list:
    globals()[bank]['사용자리뷰']=globals()[bank]['사용자리뷰'].apply(text_clean)

# 은행별 긍부정 리뷰 워드클라우드 만들기

In [ ]:
!pip install wordcloud

In [ ]:
from wordcloud import WordCloud
from konlpy.tag import Mecab
mecab=Mecab()

In [ ]:
print(type(hana_pos['사용자리뷰']))
temp=hana_pos['사용자리뷰'].tolist()
print(type(temp))
temp

In [ ]:
# 사용자리뷰 컬럼을 series 에서 list로 변경해서 담고 
#다시 join으로 1개의 문자열로 변환
text_data=hana_pos['사용자리뷰'].astype(str).tolist()
full_text=" ".join(text_data)

# 불용어 세트 생성
stopwords=set(['은행','어플','뱅킹','앱','하나'])

# 명사추출
nouns=mecab.nouns(full_text)
filtered=[word for word in nouns if len(word)>1 and word not in stopwords]

# 빈도수 집계
from collections import Counter
word_freq=Counter(filtered)

# 워드클라우드 생성
wc= WordCloud(font_path="NanumGothic.ttf",
              background_color="white",
              width=800,
              height=400
             ).generate_from_frequencies(word_freq)

plt.figure(figsize=(10,5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title(f"{hana_pos['은행명'].unique()[0]}{hana_pos['긍정'].unique()[0]} 사용자리뷰")
plt.show

# 반복문으로 은행별 만족/불만족 워드클라우드 그리기
* globals()[변수명]

In [ ]:
for bank in pos_neg_list:
    text_data=globals()[bank]['사용자리뷰'].astype(str).tolist()
    full_text=" ".join(text_data)

    # 불용어 세트 생성
    stopwords=set(['은행','어플','뱅킹','앱','하나'])

    # 명사추출
    nouns=mecab.nouns(full_text)
    filtered=[word for word in nouns if len(word)>1 and word not in stopwords]

    # 빈도수 집계
    from collections import Counter
    word_freq=Counter(filtered)

    # 워드클라우드 생성
    wc= WordCloud(font_path="NanumGothic.ttf",
                  background_color="white",
                  width=800,
                  height=400
                 ).generate_from_frequencies(word_freq)

    plt.figure(figsize=(10,5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f"{globals()[bank]['은행명'].unique()[0]}{globals()[bank]['은행명'].unique()[0]} 사용자리뷰")
    plt.show

# 토픽 모델링(LDA 기반)
* LDA는 문서 집합에서 주제를 추출하는 토픽모델링 기법
* 각 문서(리뷰1개)는 여러 주제들로 구성되어 있고, 각 주제는 특정 단어들의 분포로 표현된다는 베이지안 확률에 기반

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

In [ ]:
for bank in pos_neg_list:
    if len(globals()[bank]['사용자리뷰']) < 20:
        print(f"{bank} 리뷰 수 부족 - 스킵")
        continue
    vectorizer= CountVectorizer(max_df=0.9, min_df=10)
    review_vec= vectorizer.fit_transform(globals()[bank]['사용자리뷰'])
    lda=LatentDirichletAllocation(n_components=5, random_state=42)
    lda.fit(review_vec)
    words=vectorizer.get_feature_names_out()
    print(f"{bank} 토픽별 상위 단어")
    print("="*80)
    for i, topic in enumerate(lda.components_):
        top=[words[idx] for idx in topic.argsort()[-10:]]
        print(f"토픽 #{i}: {top}\n")

# 최적 토픽 개수 구하고 리뷰에서 토픽 추출하기
* gensim 모듈 사용
* mecab을 이용해서 형태소 분리 후, 벡터화
* coherence 점수를 계산해서 최적 토픽 개수 산출

In [ ]:
#!pip install gensim

In [ ]:
from gensim.models import CoherenceModel
from gensim import corpora
from konlpy.tag import Mecab
mecab=Mecab()

In [ ]:
# mecab을 이용한 형태소 분리
def tokenize_texts(texts):
    result=[]
    for doc in texts:
        temp=[]
        for word in mecab.nouns(doc):
            if len(word)>1:
                temp.append(word)
        result.append(temp)
    return result

# 최적 토픽 수 찾기
def find_topics(texts, start=2, end=8):
    tokenized=tokenize_texts(texts)
    dictionary=corpora.Dictionary(tokenized)
    corpus=[dictionary.doc2bow(text) for text in tokenized]
    
    vc=CountVectorizer(tokenizer=lambda x: x, lowercase=False)
    doc_term_metrix=vc.fit_transform(tokenized)
    
    best_score=-1
    best_n=start
    
    
    for n_topics in range(start, end+1):
        lda_model=LatentDirichletAllocation(n_components=n_topics, random_state=42)
        lda_model.fit(doc_term_metrix)
        
        topics=[]
        for topic in lda_model.components_:
            top_words=[vc.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]]
            topics.append(top_words)
            
        cm=CoherenceModel(topics=topics, texts=tokenized, dictionary=dictionary, coherence="c_v")
        
        score=cm.get_coherence()
        print(f"토픽 수: {n_topics}, Coherence_score:{score:.4f}")
        if score > best_score:
            best_score=score
            best_n=n_topics
        return best_n
            
   

In [ ]:
for bank in pos_neg_list:
    if len(globals()[bank]['사용자리뷰']) < 20:
        print(f"{bank} 리뷰 수 부족 - 스킵")
        continue
        
    df=globals()[bank]['사용자리뷰']
    texts=df.tolist()
    
    #최적의 컴포넌트 수 찾기
    best_k=find_topics(texts)
        
    vectorizer= CountVectorizer(max_df=0.9, min_df=10)
    review_vec= vectorizer.fit_transform(df)
    lda=LatentDirichletAllocation(n_components=best_k, random_state=42)
    lda.fit(review_vec)
    words=vectorizer.get_feature_names_out()
    print(f"{bank} 토픽별 상위 단어")
    print("="*80)
    for i, topic in enumerate(lda.components_):
        top=[words[idx] for idx in topic.argsort()[-10:]]
        print(f"토픽 #{i}: {top}\n")

# LDA 기반 최적 토픽 추출 Word2vec 기반 t-SNE 시각화

In [ ]:
from gensim.models import Word2Vec
from sklearn.manifold import TSNE

In [ ]:
for bank in pos_neg_list:
    if len(globals()[bank]['사용자리뷰']) < 20:
        print(f"{bank} 리뷰 수 부족 - 스킵")
        continue
        
    df=globals()[bank]['사용자리뷰']
    texts=df.tolist()
    
    #최적의 컴포넌트 수 찾기
    best_k=find_topics(texts)
        
    vectorizer= CountVectorizer(max_df=0.9, min_df=10)
    
    try:
        review_vec= vectorizer.fit_transform(df)
        lda=LatentDirichletAllocation(n_components=best_k, random_state=42)
        lda.fit(review_vec)
        words=vectorizer.get_feature_names_out()

        all_topic_words=set()
        print(f"{bank} 토픽{best_k}별 상위 단어")
        print("="*80)
        for i, topic in enumerate(lda.components_):
            top=[words[idx] for idx in topic.argsort()[-10:]]
            print(f"토픽 #{i}: {top}\n")
            all_topic_words.update(top)

        # 전체 리뷰로 Word2Vec 학습
        all_reviews=train_df['사용자리뷰'].dropna().tolist()
        all_tokenized=tokenize
        w2v_model= Word2Vec(sentences=all_tokenized, vector_size=100, window=5, min_count=5
                           , workers=4, sg=1)

        # 토픽 단어 중 Word2Vec에 있는 것만 시각화
        valid_words=[word for word in all_topic_words if word in w2v_model.wv]
        vectors=np.array([w2v_model.wv[word] for word in valid_words])

        if len(valid_words) >=2:
            tsne= TSNE(n_components=2, random_state=42, perplexity=5, n_iter=5000)
            reduced_vecs=tsne.fit_transform(vectors)

            plt.figure(figsize=(10,6))
            for i, word in enumerate(valid_words):
                x, y=reduced_vecs[i]
                plt.scatter(x,y)
                plt.text(x+0.01, y+0.01, word, fontsize=12)
            plt.title{f"{bank} 토픽 단어의 Word2Vec 유사도 기반 시각화 t-sne"}
            plt.grid(True)
            plt.show()
        
    else:
        print("word2vec에 있는 단어가 perplexity 개수보다 작음")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from gensim.models import CoherenceModel, Word2Vec
from gensim import corpora
from konlpy.tag import Mecab
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np

mecab = Mecab()

# 형태소 기반 명사 토큰화 함수
def tokenize_texts(texts):
    return [[word for word in mecab.nouns(doc) if len(word) > 1] for doc in texts]

# 최적 토픽 수 탐색 함수
def find_optimal_topics(tokenized_texts, start=2, end=8):
    dictionary = corpora.Dictionary(tokenized_texts)
    corpus = [dictionary.doc2bow(text) for text in tokenized_texts]

    vectorizer = CountVectorizer(tokenizer=lambda x: x, lowercase=False)
    doc_term_matrix = vectorizer.fit_transform(tokenized_texts)

    best_score = -1
    best_n = start

    for n_topics in range(start, end + 1):
        try:
            lda_model = LatentDirichletAllocation(n_components=n_topics, random_state=42)
            lda_model.fit(doc_term_matrix)

            topics = []
            for topic in lda_model.components_:
                top_words = [vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]]
                topics.append(top_words)

            cm = CoherenceModel(topics=topics, texts=tokenized_texts, dictionary=dictionary, coherence='c_v')
            score = cm.get_coherence()
            print(f"  → 토픽 수: {n_topics}, Coherence Score: {score:.4f}")
            if score > best_score:
                best_score = score
                best_n = n_topics
        except:
            continue

    return best_n

# 실제 실행 코드
for var in pos_neg_list:
    df = globals()[var]
    if len(df) < 20:
        print(f"{var} 리뷰 수 부족 - 스킵")
        continue

    texts = df['사용자리뷰'].dropna().tolist()
    print(f"\n[{var}] 최적 토픽 수 계산 중...")

    tokenized_texts = tokenize_texts(texts)
    print(tokenized_texts)
    best_k = find_optimal_topics(tokenized_texts)

    vectorizer = CountVectorizer(tokenizer=lambda x: x, lowercase=False, max_df=0.9, min_df=10)

    try:
        review_vec = vectorizer.fit_transform(tokenized_texts)
        lda_model = LatentDirichletAllocation(n_components=best_k, random_state=42)
        lda_model.fit(review_vec)
        words = vectorizer.get_feature_names_out()

        all_topic_words = set()
        print(f"\n[{var}] 토픽별 상위 단어 (K={best_k})\n" + "-" * 40)
        for i, topic in enumerate(lda_model.components_):
            top = [words[idx] for idx in topic.argsort()[-10:]]
            print(f"Topic #{i}: {top}")
            all_topic_words.update(top)

        # 전체 리뷰로 Word2Vec 학습
        all_reviews = train_df['사용자리뷰'].dropna().tolist()
        all_tokenized = tokenize_texts(all_reviews)
        w2v_model = Word2Vec(sentences=all_tokenized, vector_size=100, window=5, min_count=5, workers=4, sg=1)

        # 토픽 단어 중 Word2Vec에 있는 것만 시각화
        valid_words = [word for word in all_topic_words if word in w2v_model.wv]
        vectors = np.array([w2v_model.wv[word] for word in valid_words])

        if len(valid_words) >= 2:
            tsne = TSNE(n_components=2, random_state=0, perplexity=5, n_iter=5000)
            reduced_vecs = tsne.fit_transform(vectors)

            plt.figure(figsize=(10, 6))
            for i, word in enumerate(valid_words):
                x, y = reduced_vecs[i]
                plt.scatter(x, y)
                plt.text(x + 0.01, y + 0.01, word, fontsize=12)
            plt.title(f"{var} 토픽 단어의 Word2Vec 유사도 기반 시각화 (t-SNE)")
            plt.grid(True)
            plt.show()
        else:
            print("Word2Vec 모델에 포함된 단어가 너무 적습니다.")

    except ValueError as e:
        print(f"{var} LDA 오류: {e}")


In [ ]:
wc=WordCloud(font_path='NanumGothic.ttf', background_color='white')

plt.figure(figsize=(10,5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')

In [ ]:
cm=CoherenceModel(topics=topics, texts=tokenized)